# Training Dataset Catalog

Source-of-truth reference for **public training datasets** cited by recent open models (SmolLM, OLMo, Ouro, Qwen) and MrCogito experiment plans.

| Artifact | Role |
| --- | --- |
| `analysis/training_dataset_catalog.json` | Machine-readable catalog (models + datasets) |
| `analysis/long_dataset_candidates.json` | HF load configs for sequence-length sampling |
| `playground/long_dataset_seq_len_analysis.ipynb` | 100-row seq-len charts + interpretation |
| `analysis/dataset_seqlen_distribution.py` | CPU streaming sampler |

Updated: **2026-06-18**

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "playground" else NOTEBOOK_DIR

CATALOG_PATH = REPO_ROOT / "analysis" / "training_dataset_catalog.json"
SEQLEN_SUMMARY_PATH = REPO_ROOT / "Cache" / "Evaluation_reports" / "seqlen_model_mix_100" / "seqlen_dist_summary.json"
CANDIDATES_PATH = REPO_ROOT / "analysis" / "long_dataset_candidates.json"

catalog = json.loads(CATALOG_PATH.read_text())
candidates = json.loads(CANDIDATES_PATH.read_text())
seqlen = json.loads(SEQLEN_SUMMARY_PATH.read_text()) if SEQLEN_SUMMARY_PATH.exists() else []
seqlen_by_name = {row["name"]: row for row in seqlen}

print(f"Catalog: {len(catalog['datasets'])} datasets, {len(catalog['models'])} model programs")
print(f"Measured (100-row sample): {len(seqlen)} datasets")

In [ ]:
def md_table(rows: list[dict], columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(str(r.get(c, "")) for c in columns) + " |" for r in rows]
    return "\n".join([header, sep, *body])


dataset_rows = []
for ds in catalog["datasets"]:
    meas = seqlen_by_name.get(ds.get("sample_name", ""))
    p50 = meas["stats"]["p50"] if meas else "—"
    gt4k = meas["longer_than_pct"].get("4096", "—") if meas else "—"
    gt8k = meas["longer_than_pct"].get("8192", "—") if meas else "—"
    dataset_rows.append({
        "id": ds["id"],
        "hf_id": ds["hf_id"],
        "scale": ds["scale"],
        "type": ds["type"],
        "p50": p50,
        ">4k%": gt4k,
        ">8k%": gt8k,
        "used_by": ", ".join(ds.get("used_by", [])),
    })

display(Markdown("## All cataloged datasets (with 100-row seq-len sample where measured)\n"
    + md_table(dataset_rows, ["id", "hf_id", "scale", "type", "p50", ">4k%", ">8k%", "used_by"])))


In [ ]:
for model in catalog["models"]:
    rows = []
    for ref in model.get("dataset_refs", []):
        ds = next((d for d in catalog["datasets"] if d["id"] == ref), None)
        if not ds:
            rows.append({"ref": ref, "hf_id": "(not in catalog)", "scale": "", "type": "", "notes": ""})
            continue
        rows.append({
            "ref": ref,
            "hf_id": ds["hf_id"],
            "scale": ds["scale"],
            "type": ds["type"],
            "notes": ds.get("load_notes", ""),
        })
    stages = "; ".join(f"{s['name']}: {s.get('mix', '')}" for s in model.get("stages", []))
    display(Markdown(
        f"### {model['name']} ({model.get('total_tokens', '?')})\n"
        f"Sources: {', '.join(model.get('sources', []))}\n\n"
        + (f"Stages: {stages}\n\n" if stages else "")
        + (f"*{model.get('notes', '')}*\n\n" if model.get('notes') else "")
        + md_table(rows, ["ref", "hf_id", "scale", "type", "notes"])
    ))

In [ ]:
detail_rows = []
for ds in catalog["datasets"]:
    detail_rows.append({
        "id": ds["id"],
        "domains": ds.get("domains", ""),
        "typical_length": ds.get("typical_length", ""),
        "variety": ds.get("variety", ""),
        "load_notes": ds.get("load_notes", ""),
    })

display(Markdown("## Dataset detail (length + variety + loading)\n" + md_table(detail_rows, ["id", "domains", "typical_length", "variety", "load_notes"])))

## Model-specific notes

### Ouro (LoopLM, arxiv:2510.25741)
7.7T tokens, all open-source. Stage 1: Nemotron-CC (6.3T) + MAP-CC + OpenCoder + MegaMath-web. Stage 2 CT anneal at **16k seq**. Stage 3: **ProLong-64K** (20B). Stage 4 mid-train: OpenThoughts3, AceReason, OpenCodeReasoning.

### SmolLM3 (HuggingFaceTB blog)
11.2T three-stage mix evolving from 85/12/3 (web/code/math) to 63/24/13. Key public HF IDs: FineWeb-Edu, DCLM, FineWeb-2 (multilingual), Stack-Edu, FineMath, MegaMath, OpenMathReasoning.

### OLMo 2 (allenai)
Dolmino-mix-1124 (1.124T) and olmo-mix-1124. Components include DCLM, Dolma sources, StarCoder, peS2o, arXiv, StackExchange.

### Qwen3 (arxiv:2505.09388)
~36T tokens, 119 languages. S1 general >30T, S2 +5T STEM/code/reasoning, S3 long-context (75% samples 16k–32k). Most corpus is proprietary; public HF proxies listed in catalog.

### Kimi / Moonshot
No public training corpus release as of 2026-06.

## Rerun sequence-length sampling

```bash
uv run python analysis/dataset_seqlen_distribution.py \
  --candidates analysis/long_dataset_candidates.json \
  --tokenizer HuggingFaceTB/SmolLM2-135M \
  --max_docs 1000 \
  --out_dir Cache/Evaluation_reports/seqlen_model_mix_1k
```

Then open `playground/long_dataset_seq_len_analysis.ipynb` for charts.